# Task 3 - Kafka topic and event design

Nodes, edges, metadata, and parser failures have separate topics.
Every record carries a schema version, event time, stable key, content hash,
run ID, event ID, and operation.

```mermaid
flowchart TB
  P[Parser Service] -->|node_id key| N[cpg.nodes.v1 compact]
  P -->|edge_id key| E[cpg.edges.v1 compact]
  P -->|file_id key| M[cpg.source-metadata.v1 compact]
  P -->|error_id key| X[cpg.parser-errors.v1 delete retention]
  N --> C[Neo4j Kafka Connect]
  E --> C
  M --> S[Spark Structured Streaming]
```

## Topic boundaries and replay semantics

The four required event families are separated because their consumers and
retention needs are different. Neo4j subscribes to nodes and edges, Spark reads
metadata, and parser failures remain available for diagnosis without being
treated as graph state. Node, edge, and metadata topics use stable entity keys
with compaction; the error topic instead uses time-based retention.

Every JSON envelope repeats the information needed to interpret it independently:
schema version, UTC event time, repository and file identity, run ID, content
hash, and operation. Stable keys are what allow compaction, Neo4j `MERGE`, and
MongoDB replacement upserts to converge after replay. A single multiplexed
topic would be easier to create but would force both downstream systems to
filter unrelated schemas.

For a larger platform we would prefer a Schema Registry and multiple partitions.
Here, versioned JSON Schemas and contract tests keep the environment
self-contained, while one partition per topic makes the replay sequence easy to
inspect. There is still no assumed ordering between different topics; the sink
design handles that explicitly.

In [1]:
import subprocess
from pathlib import Path

root = Path('..').resolve()
topics = [
    'cpg.nodes.v1', 'cpg.edges.v1', 'cpg.source-metadata.v1',
    'cpg.parser-errors.v1', 'cpg.neo4j-dlq.v1',
]
for topic in topics:
    result = subprocess.run(
        ['docker', 'compose', 'exec', '-T', 'broker', 'kafka-topics',
         '--bootstrap-server', 'broker:29092', '--describe', '--topic', topic],
        cwd=root, capture_output=True, text=True, check=True,
    )
    print(result.stdout.rstrip())
print('PASS: four required topics and the connector DLQ are configured')

Topic: cpg.nodes.v1	TopicId: YyY-414WRLSQ9NjuaZCY1A	PartitionCount: 1	ReplicationFactor: 1	Configs: cleanup.policy=compact
	Topic: cpg.nodes.v1	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr:


Topic: cpg.edges.v1	TopicId: tQC-F1ksTbm5HvcTgilxjQ	PartitionCount: 1	ReplicationFactor: 1	Configs: cleanup.policy=compact
	Topic: cpg.edges.v1	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr:


Topic: cpg.source-metadata.v1	TopicId: 1YXULaXYTue71lwhS8p7og	PartitionCount: 1	ReplicationFactor: 1	Configs: cleanup.policy=compact
	Topic: cpg.source-metadata.v1	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr:


Topic: cpg.parser-errors.v1	TopicId: 0MJGt2teQquWtGdKsh0UUQ	PartitionCount: 1	ReplicationFactor: 1	Configs: cleanup.policy=delete,retention.ms=604800000
	Topic: cpg.parser-errors.v1	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr:


Topic: cpg.neo4j-dlq.v1	TopicId: 8DQjWsITTjuKPshAYrF4BA	PartitionCount: 1	ReplicationFactor: 1	Configs: cleanup.policy=delete,retention.ms=604800000
	Topic: cpg.neo4j-dlq.v1	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr:
PASS: four required topics and the connector DLQ are configured


In [2]:
import json
from pathlib import Path

root = Path('..').resolve()
evidence = json.loads((root / 'evidence/runtime/verification.json').read_text(encoding='utf-8'))
samples = evidence['kafka_samples']
required = {'cpg.nodes.v1', 'cpg.edges.v1', 'cpg.source-metadata.v1', 'cpg.parser-errors.v1'}
assert required <= set(samples)
for record in samples.values():
    assert record['source'].startswith('Kafka broker')
    assert record['key']
    assert record['value']['schema_version'] == '1.0'
    assert record['value']['event_time'].endswith('Z')
print(json.dumps(samples, indent=2, ensure_ascii=False))
print('PASS: read_committed broker samples cover all four required topics')

{
  "cpg.nodes.v1": {
    "source": "Kafka broker via read_committed kafka-console-consumer",
    "key": "b6569b0fda5380bcc1f06d903ea04c6654a7795938904c1a0fce0061449165d3",
    "value": {
      "schema_version": "1.0",
      "event_time": "2026-07-20T05:56:53.556796Z",
      "repo_id": "huggingface/optimum",
      "file_id": "c6ae6f5556f2b93b05a9f09c6c2907e8b49239a4167a4f0a4e766b7b9ce4ace2",
      "run_id": "0e24e3e515da407eb97fb170c9e4fa69",
      "content_hash": "d9981103fcb1b057d82d44e24781f1e0547d9dca5212450da54eac55089a777a",
      "op": "upsert",
      "node": {
        "id": "b6569b0fda5380bcc1f06d903ea04c6654a7795938904c1a0fce0061449165d3",
        "kind": "AST",
        "ast_type": "Module",
        "file_id": "c6ae6f5556f2b93b05a9f09c6c2907e8b49239a4167a4f0a4e766b7b9ce4ace2",
        "structural_path": "$",
        "line": 0,
        "column": 0,
        "end_line": 0,
        "end_column": 0,
        "name": "",
        "value": ""
      },
      "event_id": "4be15b56cb6f996

## Reflection

Topic inspection confirmed the intended partition count,
replication factor, cleanup policies, keys, schema versions, and UTC timestamps.
Our first evidence consisted of records captured before publication, which only
proved that Python could serialize them. The replay script now publishes an
invalid fixture and then consumes `read_committed` samples back from the live
broker for all four required topics. That change turned the chapter from a
contract demonstration into evidence that Kafka actually accepted and exposed
the records.